# 01 — Neural Networks from Scratch (NumPy Only)

We build a complete neural network using **only NumPy** — no PyTorch, no TensorFlow.
By the end, you'll understand exactly what happens when data flows through a network.

## What We'll Build
1. **Neurons** — weighted sums + bias
2. **Activation functions** — ReLU, Sigmoid, Tanh (implement + visualize)
3. **Linear layer** — matrix multiplication for transforming data
4. **2-layer network** — that learns XOR (the classic "can't be solved by a single neuron" problem)
5. **Decision boundary visualization** — see what the network actually learned

---

## Key Terminology

Before we start, here are the terms you'll encounter. Each is explained in detail when first used.

| Term | Plain English | Why It Matters |
|---|---|---|
| **Neuron** | A tiny math function: multiply inputs by weights, add bias, apply activation | The basic unit everything is built from |
| **Weight** | A number that controls how much a particular input matters | The model "learns" by adjusting these |
| **Bias** | A number added after the weighted sum — shifts the output | Lets the neuron fire even when all inputs are zero |
| **Activation function** | A non-linear function applied after the weighted sum (e.g., ReLU, Sigmoid) | Without it, stacking layers is pointless — the whole network collapses to one linear equation |
| **Forward pass** | Running data through the network from input to output to get a prediction | This is the "inference" step — making a prediction |
| **Loss function** | A single number measuring how wrong the prediction is (lower = better) | Guides the learning — the network tries to minimize this |
| **MSE (Mean Squared Error)** | Average of (prediction - target)² across all samples | Simple loss function; penalizes large errors heavily |
| **Backward pass (Backprop)** | Computing how much each weight contributed to the error, then adjusting | The "learning" step — covered in detail in Notebook 02 |
| **Epoch** | One complete pass through the entire training dataset | A unit of training progress |
| **Learning rate (lr)** | How big each weight update step is (e.g., 0.01) | Too high → unstable, too low → slow learning |
| **SGD (Stochastic Gradient Descent)** | Simplest optimizer: `weight -= lr * gradient` | Every modern optimizer (AdamW, etc.) is a fancier version of this |
| **Batch size** | Number of samples processed together before one weight update | Larger = more stable gradients, smaller = faster updates |
| **Parameter** | Any learned number (weight or bias) in the network | "GPT-4 has 1.8T parameters" means 1.8 trillion learned numbers |
| **Hyperparameter** | A setting YOU choose (learning rate, hidden size, etc.) — not learned by the model | Tuning these is a big part of ML engineering |
| **Decision boundary** | The line/surface that separates classes in input space | Visualizing this shows what the network actually learned |
| **Linearly separable** | Classes that CAN be separated by a straight line | XOR is NOT linearly separable — that's why it needs 2 layers |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part 1: What is a Neuron?

A neuron computes: **output = activation(weights · inputs + bias)**

That's it. Every AI model — from the simplest to Claude — is built from variations of this.

```
input_1 ---(w1)---\
                    +--> [sum + bias] --> [activation] --> output
input_2 ---(w2)---/
```

The **weights** control how much each input matters.  
The **bias** shifts the decision boundary.  
The **activation** adds non-linearity (without it, stacking layers is pointless).

In [ ]:
def neuron(inputs, weights, bias):
    """A single neuron: weighted sum + bias."""
    return np.dot(inputs, weights) + bias

# Example: a neuron with 2 inputs
inputs = np.array([1.0, 2.0])
weights = np.array([0.5, -0.3])
bias = 0.1

output = neuron(inputs, weights, bias)
print(f"Inputs:  {inputs}")
print(f"Weights: {weights}")
print(f"Bias:    {bias}")
print(f"Output:  {inputs[0]}*{weights[0]} + {inputs[1]}*{weights[1]} + {bias} = {output}")

## Part 2: Activation Functions

Without activation functions, a neural network is just a linear transformation —  
no matter how many layers, `f(g(x)) = Ax + b` is still linear.

Activations add **non-linearity**, letting the network learn complex patterns.

We'll implement the three classics:

### Sigmoid (1990s era)
- Formula: `σ(x) = 1 / (1 + e^(-x))`
- Output range: (0, 1) — naturally represents probabilities
- Origin: Inspired by biological neurons (fires or doesn't)
- Problem: **Vanishing gradient** — for very large or small inputs, the gradient is nearly 0,
  so weights barely update. This makes deep networks learn very slowly.

### Tanh (improved sigmoid)
- Formula: `tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))`
- Output range: (-1, 1) — **zero-centered**, which helps gradient flow
- Better than sigmoid for hidden layers because outputs are centered around 0
- Still suffers from vanishing gradients at extremes

### ReLU (Rectified Linear Unit — the modern default)
- Formula: `ReLU(x) = max(0, x)`
- Origin: Introduced by Nair & Hinton (2010). Became the default after AlexNet (2012).
- Why it won: Dead simple, fast to compute, and gradients don't vanish for positive values
- **Dead neuron problem**: if a neuron's input is always negative, it never activates and never learns.
  Modern variants (Leaky ReLU, GELU, Swish) address this — we'll see Swish in Notebook 04.

In [ ]:
def sigmoid(x):
    """Squash to (0, 1). Used for binary classification output."""
    return 1.0 / (1.0 + np.exp(-x))

def tanh(x):
    """Squash to (-1, 1). Zero-centered — better than sigmoid for hidden layers."""
    return np.tanh(x)

def relu(x):
    """max(0, x). Simple and effective. The default choice for hidden layers."""
    return np.maximum(0, x)

# Visualize all three
x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, fn, name, color in [
    (axes[0], sigmoid, 'Sigmoid', '#2196F3'),
    (axes[1], tanh, 'Tanh', '#4CAF50'),
    (axes[2], relu, 'ReLU', '#FF5722'),
]:
    ax.plot(x, fn(x), color=color, linewidth=2)
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('input')
    ax.set_ylabel('output')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: ReLU is the most commonly used because:")
print("  1. Fast to compute (just a max)")
print("  2. No vanishing gradient for positive values")
print("  3. Introduces sparsity (dead neurons for negative inputs)")

## Part 3: Linear Layer

A **Linear layer** (also called **Dense** or **Fully Connected / FC**) is many neurons in parallel:

```
output = input @ W^T + b
```

Where:
- `input` has shape `(batch_size, in_features)` — a batch of samples
- `W` has shape `(out_features, in_features)` — one row per neuron
- `b` has shape `(out_features,)` — one bias per neuron
- `output` has shape `(batch_size, out_features)`

### Why "Linear"?
Because `y = Wx + b` is a linear equation. The layer itself is linear — it's the
activation function that adds non-linearity. Together: `nonlinear(Wx + b)`.

### Weight Initialization — Why It Matters
If we initialize weights randomly from a standard normal distribution, the outputs
of each layer will have wildly different magnitudes. After 10 layers, signals either
**explode** (become infinity) or **vanish** (become zero).

**Xavier/Glorot initialization** (Glorot & Bengio, 2010): `W ~ N(0, 2/(fan_in + fan_out))`
- Keeps the variance of outputs ≈ variance of inputs across layers
- Used with sigmoid/tanh activations

**He/Kaiming initialization** (He et al., 2015): `W ~ N(0, 2/fan_in)`
- Accounts for ReLU killing half the values (sets them to 0)
- Used with ReLU activations — this is what we use below

This is the fundamental building block. **Every** layer in a transformer is ultimately
built from linear layers + activations.

In [ ]:
class Linear:
    """A fully connected layer: y = x @ W^T + b
    
    This is the same as PyTorch's nn.Linear, built from scratch.
    """
    
    def __init__(self, in_features, out_features):
        # Xavier/Glorot initialization — keeps variance stable across layers
        # Without this, deep networks either explode or vanish
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(out_features, in_features) * scale
        self.b = np.zeros(out_features)
    
    def forward(self, x):
        """x shape: (batch_size, in_features) -> (batch_size, out_features)"""
        self.input = x  # save for backward pass (Phase 1.2)
        return x @ self.W.T + self.b
    
    def __repr__(self):
        return f"Linear({self.W.shape[1]} -> {self.W.shape[0]})"

# Demo: transform 2D input to 3D output
layer = Linear(2, 3)
x = np.array([[1.0, 2.0],
              [3.0, 4.0]])  # batch of 2 samples, each with 2 features

out = layer.forward(x)
print(f"Input shape:  {x.shape}  (2 samples, 2 features)")
print(f"Weight shape: {layer.W.shape}  (3 neurons, 2 inputs each)")
print(f"Output shape: {out.shape}  (2 samples, 3 features)")
print(f"\nOutput:\n{out}")
print(f"\nParameters: {layer.W.size + layer.b.size} (W: {layer.W.size}, b: {layer.b.size})")

## Part 4: The XOR Problem

XOR is the classic test for neural networks because **a single neuron cannot solve it**.

| Input A | Input B | XOR Output |
|---------|---------|------------|
| 0       | 0       | 0          |
| 0       | 1       | 1          |
| 1       | 0       | 1          |
| 1       | 1       | 0          |

A single neuron can only draw a **straight line** to separate classes.  
XOR requires a **curved** decision boundary — you need at least 2 layers.

This is why **depth matters**: stacking layers with non-linear activations  
lets networks learn arbitrarily complex functions.

In [ ]:
# XOR dataset
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=np.float64)

y = np.array([[0],
              [1],
              [1],
              [0]], dtype=np.float64)

# Visualize XOR — notice the classes are NOT linearly separable
fig, ax = plt.subplots(figsize=(5, 5))
for i in range(len(X)):
    color = '#FF5722' if y[i] == 1 else '#2196F3'
    marker = 'o' if y[i] == 1 else 's'
    ax.scatter(X[i, 0], X[i, 1], c=color, s=200, marker=marker, 
              edgecolors='black', linewidth=2, zorder=5)

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('Input A', fontsize=12)
ax.set_ylabel('Input B', fontsize=12)
ax.set_title('XOR Problem\n(no single line can separate red from blue)', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(['Class 1 (XOR=1)', 'Class 0 (XOR=0)'], loc='upper right')
plt.show()

## Part 5: Building a 2-Layer Network to Solve XOR

Architecture:
```
Input (2) → Linear(2→4) → ReLU → Linear(4→1) → Sigmoid → Output
```

- **Hidden layer** (2→4): transforms input into a space where XOR IS linearly separable
- **ReLU**: adds non-linearity between layers
- **Output layer** (4→1): makes the final prediction
- **Sigmoid**: squashes output to (0,1) for binary classification

We'll train with:
- **MSE loss**: `L = mean((prediction - target)^2)`
- **Manual backpropagation**: compute gradients by hand (we'll formalize this in notebook 02)
- **SGD**: `weight -= learning_rate * gradient`

In [ ]:
class XORNetwork:
    """A 2-layer neural network that solves XOR.
    
    Architecture: Input(2) -> Linear(4) -> ReLU -> Linear(1) -> Sigmoid
    
    Every computation is explicit — no magic, no autograd.
    """
    
    def __init__(self, hidden_size=4):
        # Layer 1: input (2) -> hidden (hidden_size)
        scale1 = np.sqrt(2.0 / 2)  # He initialization for ReLU
        self.W1 = np.random.randn(hidden_size, 2) * scale1
        self.b1 = np.zeros(hidden_size)
        
        # Layer 2: hidden (hidden_size) -> output (1)
        scale2 = np.sqrt(2.0 / hidden_size)
        self.W2 = np.random.randn(1, hidden_size) * scale2
        self.b2 = np.zeros(1)
    
    def forward(self, X):
        """Forward pass — compute prediction step by step."""
        # Layer 1: linear + ReLU
        self.z1 = X @ self.W1.T + self.b1       # (batch, hidden)
        self.a1 = np.maximum(0, self.z1)          # ReLU
        
        # Layer 2: linear + sigmoid
        self.z2 = self.a1 @ self.W2.T + self.b2  # (batch, 1)
        self.a2 = 1.0 / (1.0 + np.exp(-self.z2)) # Sigmoid
        
        return self.a2
    
    def backward(self, X, y, output, lr=0.1):
        """Backward pass — compute gradients and update weights.
        
        This is the chain rule of calculus applied to each layer.
        We'll formalize this properly in notebook 02.
        """
        batch_size = X.shape[0]
        
        # --- Output layer gradients ---
        # Loss = MSE = mean((output - y)^2)
        # dL/d(output) = 2*(output - y) / batch_size
        # d(sigmoid)/d(z2) = sigmoid(z2) * (1 - sigmoid(z2)) = output * (1 - output)
        # dL/d(z2) = dL/d(output) * d(output)/d(z2)
        dz2 = (2.0 / batch_size) * (output - y) * output * (1 - output)  # (batch, 1)
        
        # Gradients for W2 and b2
        # z2 = a1 @ W2.T + b2
        # dL/dW2 = dz2.T @ a1
        # dL/db2 = sum(dz2)
        dW2 = dz2.T @ self.a1     # (1, hidden)
        db2 = np.sum(dz2, axis=0)  # (1,)
        
        # --- Hidden layer gradients ---
        # Propagate gradient back through W2
        da1 = dz2 @ self.W2        # (batch, hidden)
        
        # Gradient through ReLU: pass gradient where z1 > 0, zero elsewhere
        dz1 = da1 * (self.z1 > 0).astype(float)  # (batch, hidden)
        
        # Gradients for W1 and b1
        dW1 = dz1.T @ X            # (hidden, 2)
        db1 = np.sum(dz1, axis=0)  # (hidden,)
        
        # --- Update weights (SGD) ---
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
    
    def loss(self, output, y):
        """Mean Squared Error loss."""
        return np.mean((output - y) ** 2)

In [ ]:
# Train the network
np.random.seed(42)
net = XORNetwork(hidden_size=8)  # 8 hidden neurons (a bit more than needed, for robustness)

losses = []
n_epochs = 5000
lr = 1.0  # relatively high LR for this tiny problem

for epoch in range(n_epochs):
    # Forward pass
    output = net.forward(X)
    loss = net.loss(output, y)
    losses.append(loss)
    
    # Backward pass + update
    net.backward(X, y, output, lr=lr)
    
    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | Loss: {loss:.6f}")

print(f"\nFinal loss: {losses[-1]:.8f}")

In [ ]:
# Verify the network learned XOR
predictions = net.forward(X)

print("XOR Truth Table vs Network Predictions:")
print("-" * 45)
print(f"{'Input A':>8} {'Input B':>8} {'Target':>8} {'Predicted':>10}")
print("-" * 45)
for i in range(len(X)):
    print(f"{X[i,0]:>8.0f} {X[i,1]:>8.0f} {y[i,0]:>8.0f} {predictions[i,0]:>10.4f}")

# Round predictions to nearest integer for comparison
rounded = np.round(predictions)
correct = np.all(rounded == y)
print(f"\nAll correct: {correct}")

In [ ]:
# Plot the loss curve — this is what "learning" looks like
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color='#2196F3', linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.axhline(y=0.01, color='red', linestyle='--', alpha=0.5, label='0.01 threshold')
ax.legend()
plt.tight_layout()
plt.show()

print("Notice how loss drops quickly at first, then slows down.")
print("This is typical: easy patterns are learned first, fine details take longer.")

## Part 6: Visualizing the Decision Boundary

This is the most important visualization in this notebook. It shows **what the network actually learned** — the regions of input space it classifies as 0 vs 1.

For XOR, the network must create a **non-linear** boundary (impossible with 1 layer).

In [ ]:
# Create a grid of points covering the input space
resolution = 200
x_range = np.linspace(-0.5, 1.5, resolution)
y_range = np.linspace(-0.5, 1.5, resolution)
xx, yy = np.meshgrid(x_range, y_range)

# Predict for every point on the grid
grid_points = np.column_stack([xx.ravel(), yy.ravel()])
grid_predictions = net.forward(grid_points).reshape(xx.shape)

# Plot the decision boundary
fig, ax = plt.subplots(figsize=(7, 7))

# Color map showing the network's output (probability)
contour = ax.contourf(xx, yy, grid_predictions, levels=50, cmap='RdBu_r', alpha=0.8)
plt.colorbar(contour, ax=ax, label='Network output (probability)')

# Decision boundary at 0.5
ax.contour(xx, yy, grid_predictions, levels=[0.5], colors='black', linewidths=2)

# Plot the XOR points
for i in range(len(X)):
    color = '#FF5722' if y[i] == 1 else '#2196F3'
    marker = 'o' if y[i] == 1 else 's'
    ax.scatter(X[i, 0], X[i, 1], c=color, s=250, marker=marker,
              edgecolors='white', linewidth=2.5, zorder=5)

ax.set_xlabel('Input A', fontsize=12)
ax.set_ylabel('Input B', fontsize=12)
ax.set_title('Decision Boundary Learned by the Network\n'
             '(black line = decision boundary at 0.5)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print("The network learned a non-linear decision boundary!")
print("Red regions (XOR=1): (0,1) and (1,0) — different inputs")
print("Blue regions (XOR=0): (0,0) and (1,1) — same inputs")

## Part 7: What the Hidden Layer Learned

The key insight is HOW the network solves XOR:  
The hidden layer **transforms** the input into a new space where the problem IS linearly separable.

Let's look at what the hidden layer outputs for each XOR input.

In [ ]:
# Run forward pass and inspect hidden activations
_ = net.forward(X)
hidden_activations = net.a1  # shape: (4, 8) — 4 samples, 8 hidden neurons

print("Hidden layer activations (after ReLU):")
print(f"Shape: {hidden_activations.shape} (4 samples, 8 hidden neurons)\n")

labels = ['(0,0)->0', '(0,1)->1', '(1,0)->1', '(1,1)->0']
for i, label in enumerate(labels):
    print(f"{label}: [{', '.join(f'{v:.2f}' for v in hidden_activations[i])}]")

print("\nNotice: the network has found a representation where")
print("the two '1' outputs have similar patterns, and")
print("the two '0' outputs have different patterns from the '1's.")
print("This is what allows the second layer to separate them with a line.")

In [ ]:
# Visualize hidden activations as a heatmap
fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(hidden_activations, cmap='YlOrRd', aspect='auto')
ax.set_yticks(range(4))
ax.set_yticklabels(labels)
ax.set_xlabel('Hidden Neuron Index', fontsize=12)
ax.set_title('Hidden Layer Activations per XOR Input', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Activation value')

# Add value annotations
for i in range(hidden_activations.shape[0]):
    for j in range(hidden_activations.shape[1]):
        val = hidden_activations[i, j]
        color = 'white' if val > hidden_activations.max() * 0.5 else 'black'
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.show()

## Part 8: Counting Parameters

Every weight and bias is a **parameter** — a number the network learned during training.

When people say "GPT-4 has 1.8 trillion parameters" or "SmolLM2 has 360 million parameters",
they mean this exact count of weights and biases.

More parameters = more capacity to learn = needs more data and compute.

In [ ]:
def count_parameters(net):
    """Count total trainable parameters."""
    params = {
        'W1': net.W1.size,
        'b1': net.b1.size,
        'W2': net.W2.size,
        'b2': net.b2.size,
    }
    for name, count in params.items():
        print(f"  {name}: {net.__dict__[name].shape} = {count} parameters")
    total = sum(params.values())
    print(f"  Total: {total} parameters")
    return total

print("Our XOR network:")
total = count_parameters(net)

print(f"\nFor comparison:")
print(f"  Our XOR net:       {total:>15,} parameters")
print(f"  Our Phase 3 model: {'~15,000,000':>15s} parameters")
print(f"  SmolLM2-360M:      {'~360,000,000':>15s} parameters")
print(f"  GPT-4 (rumored):   {'~1,800,000,000,000':>15s} parameters")
print(f"\nSame building blocks (Linear + Activation), just scaled up enormously.")

## Summary & Key Takeaways

1. **A neuron** = weighted sum + bias + activation. That's it.
2. **Activation functions** add non-linearity — without them, deep networks collapse to a single layer.
3. **Linear layers** are just matrix multiplication — the fundamental building block.
4. **Depth matters**: a single layer can't solve XOR. Two layers with non-linearity can.
5. **Training** = forward pass (predict) → compute loss → backward pass (gradients) → update weights.
6. **Everything scales**: our 41-parameter XOR net and a trillion-parameter LLM use the same principles.

### What's Next
In **Notebook 02 (Backpropagation)**, we'll formalize the backward pass:
- Build a proper computational graph
- Implement `.backward()` for every operation
- Verify gradients with numerical checks
- This is exactly what PyTorch's autograd does under the hood.